# SecureSpeak — Verification Pass

Two checks that close the last gaps a reviewer can reach for, plus one optional experiment.

| # | Check | Why it matters |
|---|---|---|
| 1 | **SMS leakage audit** | The paper argues URL numbers are only credible under a domain-disjoint split. A reviewer will immediately ask the same of the SMS corpus. SMS datasets are full of templated near-duplicates; if they straddle the split, 98.20% is inflated the same way. This is currently the softest point in the paper *because of our own framing*. |
| 2 | **SHAP on the reported model** | Section 4.2 claims SHAP shows the model uses the MFS features. That SHAP output came from an earlier 35k run; the paper's numbers come from the 150k domain-disjoint model. Different model, so the claim needs re-grounding on the one actually reported. |
| 3 | *(optional)* Cross-dataset transfer | Answers "limited cross-dataset testing" directly. |

Pipeline settings below are copied from your blackbook: `paraphrase-multilingual-MiniLM-L12-v2`,
the same six handcrafted features, `LogisticRegression(max_iter=500, C=1.0, solver='saga',
class_weight='balanced')`, seed 42.

## 0 · Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, glob

SEED = 42
BASE = '/content/drive/MyDrive/cse498R/Datasets'
OUT_DIR = '/content/drive/MyDrive/cse498R/results_experiments'
os.makedirs(OUT_DIR, exist_ok=True)

STEALTH_CSV = os.path.join(BASE, 'StealthPhisher2025.csv')

# Bangla SMS corpus — auto-locate if the exact filename differs
BANGLA_CSV = None
for pat in ['*angala*arta*.csv', '*angla*sms*.csv', '*smish*.csv', '*sms*.csv']:
    hits = glob.glob(os.path.join(BASE, pat))
    if hits:
        BANGLA_CSV = hits[0]; break

print('StealthPhisher :', os.path.exists(STEALTH_CSV), STEALTH_CSV)
print('Bangla SMS     :', BANGLA_CSV)
if BANGLA_CSV is None:
    print('\n  Not found automatically. Files in BASE:')
    for f in sorted(os.listdir(BASE)): print('   ', f)
    print('\n  Set BANGLA_CSV manually and re-run this cell.')

StealthPhisher : True /content/drive/MyDrive/cse498R/Datasets/StealthPhisher2025.csv
Bangla SMS     : /content/drive/MyDrive/cse498R/Datasets/BangalaBarta bangla_spam_sms smishing.csv


In [3]:
!pip -q install tldextract shap sentence-transformers 2>/dev/null

import re, math, json, hashlib, warnings
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             recall_score, precision_score, confusion_matrix,
                             classification_report)

np.random.seed(SEED)
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)

RESULTS = {}

def show(title, rows):
    df = pd.DataFrame(rows)
    print('\n' + '='*len(title)); print(title); print('='*len(title))
    print(df.to_string(index=False))
    return df

def flag_if_too_good(name, acc, auc=None):
    if acc is not None and acc > 0.995:
        print(f'  !! {name}: accuracy {acc:.4f} > 0.995 — audit before reporting.')
    if auc is not None and auc > 0.999:
        print(f'  !! {name}: AUC {auc:.4f} > 0.999 — audit before reporting.')

print('Ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 5.7 MB/s eta 0:00:00
Ready.


## 1 · SMS leakage audit

The concern is templated messages. A promotional or smishing SMS is often sent thousands of
times with only the amount, date or reference number changed. Under a random split, a message
in test can be character-for-character identical to one in training apart from its digits, and
the classifier is then being scored on memorisation.

The audit proceeds in three steps: normalise each message so that trivial variation collapses,
cluster messages that are effectively the same, and then re-split so that no cluster spans the
train/test boundary. The gap between the two splits is the leakage.

In [4]:
df_bn = pd.read_csv(BANGLA_CSV, encoding='utf-8')
df_bn.columns = df_bn.columns.str.strip().str.lower()
print('Shape:', df_bn.shape)
print('Columns:', df_bn.columns.tolist())

text_col  = next((c for c in ['message','text','sms','content','msg'] if c in df_bn.columns), df_bn.columns[0])
label_col = next((c for c in ['label','class','spam','is_spam','is_phishing','type','category'] if c in df_bn.columns), None)
print(f'\nText column : {text_col}')
print(f'Label column: {label_col}')

df_bn['text_clean'] = df_bn[text_col].astype(str).str.strip()
print('\nRaw label distribution:')
print(df_bn[label_col].value_counts().to_string())
print('\nSample rows:')
print(df_bn[[text_col, label_col]].head(3).to_string())

Shape: (2772, 2)
Columns: ['label', 'text']

Text column : text
Label column: label

Raw label distribution:
label
smish     924
promo     924
normal    924

Sample rows:
                                                                      text  label
0         সোনালী ব্যাংক অ্যাকাউন্টে সমস্যা হয়েছে। কল করুন: +8801818788890  smish
1  ক্রিপ্টো বিনিয়োগে লাভবান হন! আজই শুরু করুন: http://bit.ly/CryptoInvest  smish
2        স্পেশাল ডিল শেষ দিন,৩০জিবি @৩০০৳,৩০দিন! আজই নাও- cutt.ly/jwkuSC76  promo


In [5]:
# --- Binary collapse -------------------------------------------------
# BangalaBarta has three classes: Smishing, Promotional, Normal.
# Set which classes count as positive. The blackbook reports a binary task with
# target_names ['Legit', 'Phishing/Spam'], i.e. smishing + promotional as positive.
# Change POSITIVE_CLASSES if your paper defines it differently.

POSITIVE_CLASSES = ['smish', 'smishing', 'spam', 'promo', 'promotional']

def to_binary(v):
    s = str(v).strip().lower()
    if s in ('1', 'true'):  return 1
    if s in ('0', 'false'): return 0
    return 1 if any(p in s for p in POSITIVE_CLASSES) else 0

df_bn['y'] = df_bn[label_col].map(to_binary)
print('Binary label distribution:')
print(df_bn['y'].value_counts().rename({0:'legit', 1:'smish/spam'}).to_string())
print(f'Positive rate: {df_bn["y"].mean():.4f}')
print('\nMapping actually applied (verify this matches your paper):')
print(pd.crosstab(df_bn[label_col], df_bn['y']).to_string())

Binary label distribution:
y
smish/spam    1848
legit          924
Positive rate: 0.6667

Mapping actually applied (verify this matches your paper):
y         0    1
label           
normal  924    0
promo     0  924
smish     0  924


In [6]:
# --- Normalisation: collapse trivial variation ------------------------
URL_RE   = re.compile(r'(https?://\S+|www\.\S+)')
NUM_RE   = re.compile(r'\d+')
PUNCT_RE = re.compile(r'[^\w\s\u0980-\u09FF]')   # keep Bangla block
WS_RE    = re.compile(r'\s+')

def normalise(t):
    t = str(t).lower()
    t = URL_RE.sub(' <url> ', t)
    t = NUM_RE.sub('<num>', t)
    t = PUNCT_RE.sub(' ', t)
    return WS_RE.sub(' ', t).strip()

df_bn['norm'] = df_bn['text_clean'].map(normalise)

n_exact_raw  = df_bn['text_clean'].duplicated().sum()
n_exact_norm = df_bn['norm'].duplicated().sum()

print(f'Messages                        : {len(df_bn):,}')
print(f'Exact duplicates (raw text)     : {n_exact_raw:,}  ({n_exact_raw/len(df_bn):.2%})')
print(f'Duplicates after normalisation  : {n_exact_norm:,}  ({n_exact_norm/len(df_bn):.2%})')
print(f'Distinct normalised messages    : {df_bn["norm"].nunique():,}')

if n_exact_norm:
    print('\nMost repeated templates after normalisation:')
    for tmpl, cnt in Counter(df_bn['norm']).most_common(5):
        print(f'  x{cnt:4d}  {tmpl[:90]}')

Messages                        : 2,772
Exact duplicates (raw text)     : 1,566  (56.49%)
Duplicates after normalisation  : 1,677  (60.50%)
Distinct normalised messages    : 1,095

Most repeated templates after normalisation:
  x  45  num জিবি num মিনিট num টাকা num দিন ডায়াল num num বা url
  x  37  বোনাস সহ num জিবি num টাকা num দিন ডায়াল num num বা url
  x  27  নতুন সিজনে সব পোশাকে num ডিসকাউন্ট আজই শপিং করুন
  x  20  ফ্ল্যাট num ডিসকাউন্ট সব শপিংয়ে কোড discount num
  x  20  ডাবলসেঞ্চুরি ক্যাশব্যাক ৳ num num জিবি num মি ৳ num num দিন cutt ly fwqhpkhc


In [7]:
# --- Fuzzy clustering on character n-grams ----------------------------
# Catches templates that survive normalisation with small edits.
SIM_THRESHOLD = 0.90

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1)
M = vec.fit_transform(df_bn['norm'])

# Union-Find over pairs above threshold
parent = list(range(len(df_bn)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[max(ra,rb)] = min(ra,rb)

# seed clusters with exact normalised matches
by_norm = defaultdict(list)
for i, s in enumerate(df_bn['norm']): by_norm[s].append(i)
for idxs in by_norm.values():
    for j in idxs[1:]: union(idxs[0], j)

# then fuzzy pass in blocks (2,772 msgs -> full similarity matrix is fine)
S = (M @ M.T).toarray()
np.fill_diagonal(S, 0.0)
pairs = np.argwhere(S >= SIM_THRESHOLD)
for a, b in pairs:
    if a < b: union(int(a), int(b))

df_bn['cluster'] = [find(i) for i in range(len(df_bn))]
n_clusters = df_bn['cluster'].nunique()
sizes = df_bn['cluster'].value_counts()

print(f'Similarity threshold            : {SIM_THRESHOLD}')
print(f'Near-duplicate pairs found      : {len(pairs)//2:,}')
print(f'Distinct message clusters       : {n_clusters:,} (from {len(df_bn):,} messages)')
print(f'Messages in multi-member cluster: {int((sizes[sizes>1]).sum()):,}')
print(f'Largest cluster                 : {int(sizes.max())} messages')

if n_clusters < len(df_bn):
    print('\n>> Near-duplicates present. A random split will place members of the same')
    print('   cluster in both train and test. Section 1.3 measures the effect.')
else:
    print('\n>> No near-duplicates detected. The random split carries no cluster leakage,')
    print('   and the reported SMS accuracy stands as-is.')
# --- Sensitivity guard -------------------------------------------------
# Char n-gram similarity is aggressive on short messages. If clustering collapses
# the corpus too far, the disjoint split stops being representative.
frac = n_clusters / len(df_bn)
print(f'\nClusters as fraction of corpus: {frac:.3f}')
if frac < 0.10:
    print('  !! Clustering is very aggressive (<10% of corpus remains distinct).')
    print('     Raise SIM_THRESHOLD to 0.95 and re-run this cell before trusting the split.')
elif frac < 0.30:
    print('  Note: substantial merging. Sanity-check a few clusters below, and consider')
    print('  reporting results at both 0.90 and 0.95 to show the finding is not threshold-driven.')
else:
    print('  Clustering looks reasonable.')

# Inspect the largest cluster to confirm members really are near-duplicates
big = sizes.index[0]
print(f'\nSample from the largest cluster (id={big}, n={int(sizes.iloc[0])}):')
for t in df_bn.loc[df_bn['cluster']==big, 'text_clean'].head(4):
    print('  -', str(t)[:100])


Similarity threshold            : 0.9
Near-duplicate pairs found      : 8,029
Distinct message clusters       : 1,028 (from 2,772 messages)
Messages in multi-member cluster: 2,356
Largest cluster                 : 48 messages

>> Near-duplicates present. A random split will place members of the same
   cluster in both train and test. Section 1.3 measures the effect.

Clusters as fraction of corpus: 0.371
  Clustering looks reasonable.

Sample from the largest cluster (id=44, n=48):
  - ৫জিবি+১৫০মিনিট ১৩০টাকা (৩০দিন),ডায়াল *১২১*৫২০৫# বা https://mygp.li/a0
  - ৫জিবি+১৫০মিনিট ১৩০টাকা (৩০দিন),ডায়াল *১২১*৫২০৫# বা https://mygp.li/a0
  - ৫জিবি+১৫০মিনিট ১৩০টাকা (৩০দিন),ডায়াল *১২১*৫২০৫# বা https://mygp.li/a0
  - ৫জিবি+১৫০মিনিট ১৩০টাকা (৩০দিন),ডায়াল *১২১*৫২০৫# বা https://mygp.li/a0


In [8]:
# --- Embeddings (computed once, reused for both splits) ---------------
import torch
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
mlm = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device=str(device))

def sms_features(text):
    t = str(text)
    return [
        1 if any(w in t.lower() for w in ['http','www','.com','.net','.xyz']) else 0,
        1 if any(w in t.lower() for w in ['bkash','nagad','rocket','bank','account']) else 0,
        1 if any(w in t.lower() for w in ['win','prize','free','offer','click','urgent','\u0986\u09aa\u09a8\u09be\u09b0','\u099f\u09be\u0995\u09be']) else 0,
        min(len(t)/500, 1.0),
        t.count('!') / max(len(t),1),
        sum(c.isdigit() for c in t)/max(len(t),1),
    ]

feats = np.array([sms_features(t) for t in df_bn['text_clean']])
texts = df_bn['text_clean'].tolist()
embs = []
BATCH = 256
for i in tqdm(range(0, len(texts), BATCH), desc='Embedding'):
    embs.append(mlm.encode(texts[i:i+BATCH], show_progress_bar=False, device=str(device)))
embs = np.concatenate(embs)

X_sms = np.concatenate([embs, feats], axis=1)
y_sms = df_bn['y'].values
clu   = df_bn['cluster'].values
print('Feature matrix:', X_sms.shape)

Device: cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding:   0%|          | 0/11 [00:00<?, ?it/s]

Feature matrix: (2772, 390)


In [9]:
def fit_eval(tr, te, name):
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_sms[tr]); Xte = sc.transform(X_sms[te])
    clf = LogisticRegression(max_iter=500, C=1.0, random_state=SEED,
                             class_weight='balanced', solver='saga').fit(Xtr, y_sms[tr])
    pred  = clf.predict(Xte)
    proba = clf.predict_proba(Xte)[:, 1]
    acc = accuracy_score(y_sms[te], pred)
    auc = roc_auc_score(y_sms[te], proba)
    flag_if_too_good(name, acc, auc)
    tn, fp, fn, tp = confusion_matrix(y_sms[te], pred, labels=[0,1]).ravel()
    return dict(split=name, n_test=len(te),
                accuracy=round(acc,4),
                f1=round(f1_score(y_sms[te], pred, average='weighted'),4),
                auc=round(auc,4),
                recall=round(recall_score(y_sms[te], pred, zero_division=0),4),
                fnr=round(fn/(fn+tp) if (fn+tp) else float('nan'),4))

rows = []

# Random split — the protocol behind the reported 98.20%
idx = np.arange(len(y_sms))
tr_r, te_r = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y_sms)
overlap_r = len(set(clu[tr_r]) & set(clu[te_r]))
r1 = fit_eval(tr_r, te_r, 'Random (as reported)'); r1['cluster_overlap'] = overlap_r
rows.append(r1)
print(f'Random split: {overlap_r:,} message clusters span train and test.')

# Cluster-disjoint split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_c, te_c = next(gss.split(X_sms, y_sms, groups=clu))
assert len(set(clu[tr_c]) & set(clu[te_c])) == 0, 'cluster-disjoint split failed'
r2 = fit_eval(tr_c, te_c, 'Cluster-disjoint'); r2['cluster_overlap'] = 0
rows.append(r2)
print('Cluster-disjoint split: 0 overlap (asserted).')

tbl_sms = show('TABLE — Bangla SMS detector under two split protocols', rows)
RESULTS['sms_splits'] = tbl_sms
tbl_sms.to_csv(f'{OUT_DIR}/exp4_sms_splits.csv', index=False)

Random split: 299 message clusters span train and test.
Cluster-disjoint split: 0 overlap (asserted).

TABLE — Bangla SMS detector under two split protocols
               split  n_test  accuracy     f1    auc  recall    fnr  cluster_overlap
Random (as reported)     555    0.9784 0.9784 0.9981  0.9811 0.0189              299
    Cluster-disjoint     543    0.9595 0.9591 0.9800  0.9889 0.0111                0


In [10]:
s = RESULTS['sms_splits']
a_rand = float(s.iloc[0]['accuracy']); a_clus = float(s.iloc[1]['accuracy'])
drop = a_rand - a_clus
print(f'Random split accuracy        : {a_rand:.4f}')
print(f'Cluster-disjoint accuracy    : {a_clus:.4f}')
print(f'Drop attributable to leakage : {drop:.4f}  ({drop/a_rand:.2%} relative)\n')

if drop > 0.05:
    print('LARGE DROP. The reported 98.20% was substantially template memorisation.')
    print('Report the cluster-disjoint number as primary and revise Section 4.4, including')
    print('the comparison against Tanbhir et al. — note that their number is on a random')
    print('split too, so the fair statement is that both are subject to the same effect.')
elif drop > 0.01:
    print('MODERATE DROP. Report both protocols. The gap is a finding worth a short paragraph,')
    print('and it is consistent with the leakage argument already made for URLs.')
else:
    print('SMALL DROP. The SMS detector is not relying on template memorisation.')
    print('Report both. This closes the obvious follow-up question to Section 4.1 and makes')
    print('the leakage-awareness of the paper consistent across both text modalities.')

Random split accuracy        : 0.9784
Cluster-disjoint accuracy    : 0.9595
Drop attributable to leakage : 0.0189  (1.93% relative)

MODERATE DROP. Report both protocols. The gap is a finding worth a short paragraph,
and it is consistent with the leakage argument already made for URLs.


## 2 · SHAP on the model the paper actually reports

Section 4.2 states that SHAP shows the model attending to the MFS knowledge features. That
needs to be computed on the domain-disjoint 150k model whose numbers appear in Table 1, not
on an earlier run. This section rebuilds that exact model and recomputes the attribution.

In [11]:
import tldextract

HIGH_RISK_TLDS = {'tk','ml','ga','cf','gq','pw','top','xyz','online','site','club',
                  'live','shop','info','biz','link','click','download','stream'}
FREE_HOST_TLDS = {'tk','ml','ga','cf','gq','pw'}
FINANCIAL_KW   = ['bank','login','secure','verify','update','account','password','signin',
                  'bkash','nagad','rocket','paypal','amazon','netflix','microsoft','apple','google','confirm']
BRAND_KW       = ['paypal','amazon','google','facebook','apple','microsoft','bkash','nagad','rocket']

def shannon_entropy(s):
    if not s: return 0.0
    f = {}
    for ch in s: f[ch] = f.get(ch,0)+1
    n = len(s)
    return -sum((v/n)*math.log2(v/n) for v in f.values())

def engineer_url_features(url):
    url = str(url).strip().lower()
    ext = tldextract.extract(url)
    domain, suffix, subdomain = ext.domain, ext.suffix, ext.subdomain
    path  = re.sub(r'https?://[^/]+', '', url)
    query = path.split('?',1)[1] if '?' in path else ''
    return [
        min(len(url)/500,1.0), min(url.count('.')/10,1.0), min(url.count('/')/15,1.0),
        min(len(re.findall(r'[-_@!%&=+]',url))/20,1.0),
        sum(c.isdigit() for c in url)/max(len(url),1),
        sum(c.isalpha() for c in url)/max(len(url),1),
        1.0 if suffix in HIGH_RISK_TLDS else 0.0,
        min(subdomain.count('.')+1 if subdomain else 0,5)/5,
        min(len(domain)/30,1.0),
        1.0 if re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$', url.split('/')[2] if '/' in url else url) else 0.0,
        min(sum(b in domain for b in BRAND_KW),3)/3,
        min(len(path)/200,1.0), min(len([s for s in path.split('/') if s])/10,1.0),
        1.0 if '?' in url else 0.0, min(len(query)/200,1.0),
        1.0 if url.startswith('https') else 0.0, 1.0 if 'https' in path else 0.0,
        shannon_entropy(url)/6.0, shannon_entropy(domain)/4.0,
        min(sum(k in url for k in FINANCIAL_KW),5)/5,
        1.0 if re.search(r'@|//.*@', url) else 0.0,
        min(url.count('-')/8,1.0),
        1.0 if len(url) > 75 and not url.startswith('https') else 0.0,
        1.0 if suffix in FREE_HOST_TLDS else 0.0,
        min(len(re.findall(r'\d{3,}',url))/3,1.0),
        (1.0 if url.startswith('https') else 0.0)*(0.0 if suffix in HIGH_RISK_TLDS else 1.0),
    ]

URL_FEAT_NAMES = ['url_length','dot_count','slash_count','special_chars','digit_ratio','letter_ratio',
 'high_risk_tld','subdomain_depth','domain_length','uses_ip','brand_impersonation','path_length',
 'path_segments','has_query','query_length','has_https','https_in_path','url_entropy','domain_entropy',
 'financial_kw','at_in_url','hyphen_count','long_http','free_hosting_tld','long_numbers','https_x_safe_tld']
MFS_FEATURES = ['brand_impersonation','financial_kw','free_hosting_tld','high_risk_tld']
print('Feature engineer ready.')

Feature engineer ready.


In [12]:
PHISH_W = {'1','phishing','phish','malicious','bad','spam','fake','scam','fraud','unsafe'}
LEGIT_W = {'0','legitimate','legit','benign','good','safe','real','ham','clean','normal'}
def norm_label(v):
    s = str(v).strip().lower()
    if s in PHISH_W: return 1
    if s in LEGIT_W: return 0
    try: return int(float(s) > 0.5)
    except Exception: return np.nan

chunks, N_TARGET = [], 150_000
for ch in pd.read_csv(STEALTH_CSV, chunksize=50_000, low_memory=False):
    ch.columns = ch.columns.str.strip()
    uc = next((c for c in ['url','URL','link','Link','website','Website'] if c in ch.columns), None)
    lc = next((c for c in ['label','Label','class','Class','phishing','is_phishing','target',
                           'Target','status','type','result'] if c in ch.columns), None)
    sub = ch[[uc,lc]].rename(columns={uc:'url', lc:'label'})
    sub['label'] = sub['label'].map(norm_label)
    sub = sub.dropna(subset=['label','url']); sub['label'] = sub['label'].astype(int)
    chunks.append(sub)
    if sum(len(c) for c in chunks) >= N_TARGET: break

df_sp = pd.concat(chunks, ignore_index=True).iloc[:N_TARGET].reset_index(drop=True)
df_sp['_regdom'] = [f'{tldextract.extract(u).domain}.{tldextract.extract(u).suffix}'.strip('.').lower()
                    for u in tqdm(df_sp['url'], desc='regdom')]
X_url = np.array([engineer_url_features(u) for u in tqdm(df_sp['url'], desc='URL feats')])
y_url = df_sp['label'].values
grp   = df_sp['_regdom'].values
print('Loaded:', df_sp.shape, '| features:', X_url.shape)

regdom:   0%|          | 0/150000 [00:00<?, ?it/s]

URL feats:   0%|          | 0/150000 [00:00<?, ?it/s]

Loaded: (150000, 3) | features: (150000, 26)


In [13]:
# Rebuild the exact model reported in Table 1
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_d, te_d = next(gss.split(X_url, y_url, groups=grp))
assert len(set(grp[tr_d]) & set(grp[te_d])) == 0

pipe = Pipeline([('scale', StandardScaler()),
                 ('clf', RandomForestClassifier(300, random_state=SEED, n_jobs=-1,
                                                class_weight='balanced'))]).fit(X_url[tr_d], y_url[tr_d])
acc = accuracy_score(y_url[te_d], pipe.predict(X_url[te_d]))
auc = roc_auc_score(y_url[te_d], pipe.predict_proba(X_url[te_d])[:,1])
print(f'Rebuilt domain-disjoint model: accuracy {acc:.4f}, AUC {auc:.4f}')
print('(Should match Table 1: 0.9919 / 0.9976)')

Rebuilt domain-disjoint model: accuracy 0.9919, AUC 0.9976
(Should match Table 1: 0.9919 / 0.9976)


In [14]:
import shap

# TreeExplainer on the scaled space the forest actually sees
scaler = pipe.named_steps['scale']
rf     = pipe.named_steps['clf']

SAMPLE = 2000
rng = np.random.default_rng(SEED)
sub_idx = rng.choice(te_d, size=min(SAMPLE, len(te_d)), replace=False)
X_sub = scaler.transform(X_url[sub_idx])

explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X_sub, check_additivity=False)
if isinstance(sv, list):
    sv = sv[1]
elif sv.ndim == 3:
    sv = sv[:, :, 1]

mean_abs = np.abs(sv).mean(axis=0)
order = np.argsort(mean_abs)[::-1]

rows = [dict(rank=r+1,
             feature=URL_FEAT_NAMES[i],
             mean_abs_shap=round(float(mean_abs[i]), 5),
             is_mfs_feature='yes' if URL_FEAT_NAMES[i] in MFS_FEATURES else '')
        for r, i in enumerate(order)]

tbl_shap = show(f'TABLE — SHAP attribution, domain-disjoint model (n={len(sub_idx):,} test samples)',
                rows[:12])
RESULTS['shap'] = pd.DataFrame(rows)
RESULTS['shap'].to_csv(f'{OUT_DIR}/exp5_shap.csv', index=False)

print('\nRank of each MFS knowledge feature:')
for f in MFS_FEATURES:
    r = [x for x in rows if x['feature'] == f][0]
    print(f"  {f:22s} rank {r['rank']:2d} / 26   mean|SHAP| = {r['mean_abs_shap']:.5f}")


TABLE — SHAP attribution, domain-disjoint model (n=2,000 test samples)
 rank          feature  mean_abs_shap is_mfs_feature
    1      slash_count        0.13571               
    2      path_length        0.11798               
    3        has_https        0.05706               
    4 https_x_safe_tld        0.04909               
    5    path_segments        0.04196               
    6      digit_ratio        0.03519               
    7     letter_ratio        0.02815               
    8    domain_length        0.02734               
    9      url_entropy        0.02510               
   10       url_length        0.02403               
   11  subdomain_depth        0.01887               
   12        dot_count        0.01291               

Rank of each MFS knowledge feature:
  brand_impersonation    rank 21 / 26   mean|SHAP| = 0.00002
  financial_kw           rank 19 / 26   mean|SHAP| = 0.00070
  free_hosting_tld       rank 18 / 26   mean|SHAP| = 0.00096
  high_risk_tld    

In [15]:
ranks = {f: [x for x in RESULTS['shap'].to_dict('records') if x['feature']==f][0]['rank']
         for f in MFS_FEATURES}
best = min(ranks.values())
print('MFS feature ranks:', ranks, '\n')

if best <= 5:
    print('At least one MFS feature is in the top 5. Section 4.2 can state this directly,')
    print('now grounded on the model reported in Table 1. Keep the wording careful: the')
    print('model ATTENDS to these features. The ablation already showed that removing them')
    print('costs no accuracy on this corpus. Both statements are true and not in tension —')
    print('the features are informative but redundant given the other 22.')
else:
    print('No MFS feature reaches the top 5 on this model. Revise Section 4.2 to report the')
    print('actual ranks rather than claiming prominence. Combined with the null ablation,')
    print('the honest reading is that the MFS features are not load-bearing for accuracy on')
    print('this corpus — which is exactly what the corpus audit in 4.3 explains.')

MFS feature ranks: {'brand_impersonation': 21, 'financial_kw': 19, 'free_hosting_tld': 18, 'high_risk_tld': 17} 

No MFS feature reaches the top 5 on this model. Revise Section 4.2 to report the
actual ranks rather than claiming prominence. Combined with the null ablation,
the honest reading is that the MFS features are not load-bearing for accuracy on
this corpus — which is exactly what the corpus audit in 4.3 explains.


## 3 · Optional — cross-dataset transfer

Trains on StealthPhisher and tests on a second corpus of raw URLs. Because the 26 features are
computed from the URL string alone, any corpus with a URL column and a label works. This
directly answers the "limited cross-dataset testing" objection.

Set `SECOND_CSV` to a second phishing corpus to run it.

In [16]:
SECOND_CSV = None      # e.g. os.path.join(BASE, 'PhiUSIIL.csv')

if SECOND_CSV is None:
    print('SECOND_CSV is None — skipped. Set it to run cross-dataset transfer.')
else:
    d2 = pd.read_csv(SECOND_CSV, low_memory=False)
    d2.columns = d2.columns.str.strip()
    uc = next((c for c in ['url','URL','link','Link','website','Website'] if c in d2.columns), None)
    lc = next((c for c in ['label','Label','class','Class','phishing','is_phishing',
                           'target','Target','status','type','result'] if c in d2.columns), None)
    if uc is None or lc is None:
        raise KeyError(f'URL/label column not found. Columns: {d2.columns.tolist()[:30]}')
    d2 = d2[[uc, lc]].rename(columns={uc:'url', lc:'label'})
    d2['label'] = d2['label'].map(norm_label)
    d2 = d2.dropna(subset=['url','label']); d2['label'] = d2['label'].astype(int)
    print(f'Second corpus: {len(d2):,} rows, positive rate {d2["label"].mean():.4f}')
    print('NOTE: check the label convention. Some corpora (e.g. PhiUSIIL) use 1 = legitimate.')

    X2 = np.array([engineer_url_features(u) for u in tqdm(d2['url'], desc='feats')])
    y2 = d2['label'].values
    pred2 = pipe.predict(X2)
    acc2 = accuracy_score(y2, pred2)
    auc2 = roc_auc_score(y2, pipe.predict_proba(X2)[:,1])
    flag_if_too_good('Cross-dataset', acc2, auc2)

    tbl_x = show('TABLE — cross-dataset transfer', [
        dict(evaluation='In-corpus (domain-disjoint)', n=len(te_d),
             accuracy=round(acc,4), auc=round(auc,4)),
        dict(evaluation='Cross-corpus (unseen dataset)', n=len(y2),
             accuracy=round(acc2,4), auc=round(auc2,4)),
    ])
    RESULTS['cross_dataset'] = tbl_x
    tbl_x.to_csv(f'{OUT_DIR}/exp6_cross_dataset.csv', index=False)
    print(f'\nTransfer gap: {acc-acc2:+.4f} accuracy.')
    print('A large gap is normal and worth reporting honestly — it bounds how far the')
    print('detector travels beyond the distribution it was fitted on.')

SECOND_CSV is None — skipped. Set it to run cross-dataset transfer.


## 4 · Export

In [17]:
def to_latex(df, cap, lab):
    return ('\\begin{table}[t]\n\\centering\n\\small\n'
            + df.to_latex(index=False, escape=True, column_format='l'+'r'*(df.shape[1]-1))
            + f'\\caption{{{cap}}}\n\\label{{tab:{lab}}}\n\\end{{table}}\n')

caps = {
 'sms_splits':    ('Bangla SMS detector under random and cluster-disjoint splits. Cluster overlap '
                   'counts near-duplicate message clusters spanning both partitions.', 'smssplits'),
 'shap':          ('SHAP attribution over the domain-disjoint URL model reported in Table 1.', 'shap'),
 'cross_dataset': ('Cross-dataset transfer of the URL detector.', 'crossdata'),
}

tex = []
for k, t in RESULTS.items():
    df_out = t.head(12) if k == 'shap' else t
    tex.append(to_latex(df_out, *caps.get(k, (k, k))))
tex = '\n'.join(tex)

with open(f'{OUT_DIR}/tables_verification.tex', 'w') as fh:
    fh.write(tex)

print('Written to', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print('  ', f)
print('\n' + '='*60 + '\n' + tex)

Written to /content/drive/MyDrive/cse498R/results_experiments
   exp1_splits.csv
   exp2_ablation.csv
   exp4_sms_splits.csv
   exp5_shap.csv
   run_metadata.json
   tables.tex
   tables_verification.tex

\begin{table}[t]
\centering
\small
\begin{tabular}{lrrrrrrr}
\toprule
split & n\_test & accuracy & f1 & auc & recall & fnr & cluster\_overlap \\
\midrule
Random (as reported) & 555 & 0.978400 & 0.978400 & 0.998100 & 0.981100 & 0.018900 & 299 \\
Cluster-disjoint & 543 & 0.959500 & 0.959100 & 0.980000 & 0.988900 & 0.011100 & 0 \\
\bottomrule
\end{tabular}
\caption{Bangla SMS detector under random and cluster-disjoint splits. Cluster overlap counts near-duplicate message clusters spanning both partitions.}
\label{tab:smssplits}
\end{table}

\begin{table}[t]
\centering
\small
\begin{tabular}{lrrr}
\toprule
rank & feature & mean\_abs\_shap & is\_mfs\_feature \\
\midrule
1 & slash\_count & 0.135710 &  \\
2 & path\_length & 0.117980 &  \\
3 & has\_https & 0.057060 &  \\
4 & https\_x\_safe\_t

## 5 · What each outcome means for the paper

**SMS split (Section 1).** If the drop is small, you gain a sentence in Section 4.4 that closes
the most obvious follow-up to Section 4.1 and makes the paper's leakage-awareness consistent
across both text modalities. If the drop is large, the cluster-disjoint number becomes the
reported one — and note that Tanbhir et al.'s 98.47% is on a random split, so the fair framing
is that both are subject to the same effect, not that yours is worse.

**SHAP (Section 2).** Whatever the ranks are, report them as measured on the Table 1 model.
There is no tension between "the model attends to these features" and "removing them costs no
accuracy" — informative and load-bearing are different properties, and saying so precisely is
stronger than implying the first means the second.

**Cross-dataset (Section 3).** Optional, but it is the single cheapest answer to the
generalisation objection, and a large gap reported honestly reads better than no test at all.